# MCDA Python — Colab i lokalni Jupyter
Pokrenite ćelije redom. Parametre mijenjajte u konfiguraciji.

## 1. Priprema
U Google Colabu prva ćelija automatski preuzima projekt s GitHuba i instalira pakete. Lokalno otvorite folder projekta u VS Codeu ili Jupyteru. ZIP je opciona zamjena.

In [ ]:
import sys, subprocess, pathlib
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# GitHub projekt se automatski preuzima kada se notebook otvori u Google Colabu.
GITHUB_URL = 'https://github.com/WaydenDunford/MCDA-Python.git'
PROJECT_ZIP = None  # Opciono: putanja do vlastitog ZIP-a projekta.

if IN_COLAB:
    from google.colab import files
    ROOT = pathlib.Path('/content/MCDA-Python')
    if not (ROOT / 'engine.py').exists():
        if PROJECT_ZIP:
            import zipfile
            with zipfile.ZipFile(PROJECT_ZIP) as archive:
                archive.extractall('/content')
        else:
            subprocess.check_call(['git', 'clone', GITHUB_URL, str(ROOT)])
    elif (ROOT / '.git').exists():
        subprocess.check_call(['git', '-C', str(ROOT), 'pull', '--ff-only'])
else:
    ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()

if not (ROOT / 'engine.py').exists():
    raise FileNotFoundError(f'Projekt nije pronađen u {ROOT}.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.txt')])
sys.path.insert(0, str(ROOT))

## 2. Excel ulaz
U Colabu odaberite Excel fajl. Lokalno koristite putanju do primjera ili vlastitog fajla.

In [ ]:
if IN_COLAB:
    uploaded=files.upload()
    FILE=next(iter(uploaded))
else:
    FILE=str(ROOT/'data'/'example.xlsx')

## 3. Konfiguracija

Postavke koje mijenjaš nalaze se u ćeliji **3.2**. Po potrebi otvori **3.1** za tabelu metoda, zadataka, normalizacija i parametara.

<details>
<summary><strong>3.1 Detalji metoda i parametara — klikni za prikaz</strong></summary>

U sljedećoj ćeliji mijenjajte samo postavke; matematičke module nije potrebno uređivati. `METHOD` bira metodu, `TASK` bira šta se računa, a `NORMALIZATION` je naziv vanjske normalizacije ili `None` kada metoda koristi vlastitu formulu. Za standardne metode rangiranja težine se čitaju iz Excel reda `weights`; ako ga nema, prvo ih izračunajte jednom od metoda za težine.

| `METHOD` | `TASK` | `NORMALIZATION` | Dodatni `METHOD_PARAMS` | Excel ulaz |
|---|---|---|---|---|
| `ahp` | `weights`, `ranking` | `None` | — | recipročna matrica; za ranking zaseban list alternativa za svaki kriterij |
| `aras` | `ranking` | `None` | — | matrica odluke |
| `codas` | `ranking` | `None` | `tau` (zadano `0.02`) | matrica odluke |
| `copras` | `ranking` | `None` | — | matrica odluke |
| `critic` | `weights` | `None` | — | matrica odluke |
| `dematel` | `relationships` | `None` | — | kvadratna matrica uticaja |
| `edas` | `ranking` | `None` | — | matrica odluke |
| `entropy` | `weights` | `None` | — | matrica odluke |
| `mabac` | `ranking` | `None` | — | matrica odluke |
| `macbeth` | `ranking` | `None` | — | matrica kvalitativnih sudova |
| `mairca` | `ranking` | `None` | — | matrica odluke |
| `marcos` | `ranking` | `None` | — | matrica odluke |
| `merec` | `weights` | `None` | — | matrica odluke |
| `moora` | `ranking` | `None` | — | matrica odluke |
| `promethee2` | `ranking` | `None` | `preference_functions` | matrica odluke |
| `psi` | `weights`, `ranking` | `None` | — | matrica odluke |
| `saw` | `ranking` | `max`, `minmax`, `sum`, `vector`, `nonlinear` | `p` samo uz `nonlinear` (zadano `2`) | matrica odluke |
| `todim` | `ranking` | `None` | `theta` (zadano `1`) | matrica odluke |
| `topsis` | `ranking` | `vector` | — | matrica odluke |
| `vikor` | `ranking` | `None` | `v` (zadano `0.5`) | matrica odluke |
| `waspas` | `ranking` | `None` | `lambda_` (zadano `0.5`) | matrica odluke |

**Šta znače zadaci?** `ranking` vraća ocjene i poredak alternativa; `weights` vraća težine kriterija; `relationships` vraća DEMATEL pokazatelje uticaja. `None` u tabeli znači da metoda koristi vlastitu normalizaciju i ne prima proizvoljnu vanjsku normalizaciju. Šest samostalnih normalizacija nalazi se u `normalizations/`; logaritamska `ln` postoji za zasebne eksperimente, ali nije ponuđena kao zamjena za formule ovih metoda.

**Primjeri promjena u konfiguracijskoj ćeliji:**

```python
# TOPSIS: ostavite težine u Excel redu „weights“
METHOD = 'topsis'; TASK = 'ranking'; NORMALIZATION = 'vector'; METHOD_PARAMS = {}

# Entropija: izračun težina, bez reda „weights“
METHOD = 'entropy'; TASK = 'weights'; NORMALIZATION = None; METHOD_PARAMS = {}

# WASPAS: 70% ponderisani zbir, 30% ponderisani proizvod
METHOD = 'waspas'; TASK = 'ranking'; NORMALIZATION = None
METHOD_PARAMS = {'lambda_': 0.7}

# SAW s nelinearnom potencijalnom normalizacijom
METHOD = 'saw'; TASK = 'ranking'; NORMALIZATION = 'nonlinear'
METHOD_PARAMS = {'p': 2.0}
```

Za `promethee2` se može zadati npr. `METHOD_PARAMS = {'preference_functions': {'C1': {'type': 'usual'}, 'C2': {'type': 'linear', 'q': 0, 'p': 5}, 'C3': {'type': 'usual'}}}`. Svaki kriterij mora imati postavku; podržane funkcije su `usual`, `u_shape`, `v_shape`, `level`, `linear` i `gaussian`. Bez ovog parametra svi kriteriji koriste `usual`.

`SHEET` bira Excel list (npr. `'Criteria'` za AHP); `None` znači prvi list. Za AHP ranking listovi alternativa moraju se zvati kao kriteriji (`C1`, `C2`, ...). `SHOW_STEPS=True` prikazuje sve međukorake; `False` samo glavne rezultate. `DECIMALS` mijenja prikaz i format izvoza, ne preciznost računanja. `EXPORT_EXCEL=True` čuva sve rezultate u `results/`.

Za najbržu provjeru dostupnih kombinacija možete pokrenuti `from rules import list_methods, list_normalizations, get_method_info` i, npr., `list_normalizations('saw')`.

**Excel redovi:** `Cilj` ili `max/min` označava smjer kriterija (`+`, `-`, `max`, `min`, `benefit`, `cost`). `Težina (w)` ili `weights` označava težine. Završni redovi sa samim nulama smatraju se praznim predlošcima: čitač ih preskače uz upozorenje. Ako su to stvarne alternative, postavi `SKIP_TRAILING_ZERO_ROWS = False` u ćeliji 3.2.

</details>

### 3.2 Postavke za pokretanje

Promijeni vrijednosti u ćeliji ispod, zatim nastavi na korak 4.

In [ ]:
# PROMIJENI: metoda i zadatak iz tabele iznad
METHOD = 'saw'
TASK = 'ranking'

# PROMIJENI: None za internu normalizaciju; SAW bira jednu od pet, TOPSIS 'vector'
NORMALIZATION = 'minmax'

# PROMIJENI po potrebi: naziv Excel lista i parametri metode
SHEET = None
SKIP_TRAILING_ZERO_ROWS = True  # Preskoči samo završne redove sa svim nulama.
METHOD_PARAMS = {}  # npr. {'lambda_': 0.7} za WASPAS

# PROMIJENI: prikaz, broj decimala i Excel izvoz
SHOW_STEPS = True
DECIMALS = 6
EXPORT_EXCEL = True

## 4. Učitavanje Excela

In [ ]:
from excel import load_data
data=load_data(FILE,method=METHOD,task=TASK,sheet=SHEET,
               skip_trailing_zero_rows=SKIP_TRAILING_ZERO_ROWS)

## 5. Računanje

In [ ]:
from engine import run
result=run(METHOD,task=TASK,normalization=NORMALIZATION,**data,**METHOD_PARAMS)

## 6. Prikaz

In [ ]:
from display import show_results
show_results(result,show_steps=SHOW_STEPS,decimals=DECIMALS)

## 7. Izvoz

In [ ]:
if EXPORT_EXCEL:
    from export import export_excel
    output=export_excel(result,ROOT/'results'/f'{METHOD}_results.xlsx',decimals=DECIMALS)
    print(output)
    if IN_COLAB: files.download(str(output))